# Person Door Counter — ระบบนับคนเข้า-ออกประตูด้วย AI

## ภาพรวมโครงการ
ระบบตรวจจับและนับจำนวนคนเข้า-ออกประตูจากวิดีโอ CCTV โดยใช้:
- **YOLOv5n** — ตรวจจับและติดตามตัวคน (Detection + Tracking)
- **Color Histogram (HSV)** — ดึงลักษณะเด่นของแต่ละคนเพื่อ Re-Identification (เร็วกว่า Deep Learning 50-100 เท่า)
- **Door Zone (สี่เหลี่ยมกรอบประตู)** — ตรวจจับเฉพาะคนที่เดินเข้า/ออกพื้นที่ประตู ไม่นับคนที่ยืนอยู่ในห้อง
- **Return Detection** — จำคนที่เคยเข้ามาแล้วกลับมาซ้ำได้

## 1. ติดตั้งไลบรารีที่จำเป็น

In [1]:
!pip install ultralytics

^C


## 2. สร้าง Feature Extractor สำหรับ Re-ID (จดจำตัวตนคน)
ใช้ **Color Histogram (HSV)** แปลงภาพตัดเฉพาะคนให้เป็นเวกเตอร์ตัวเลข แทนการใช้โมเดล CNN หนักอย่าง ResNet50

**ทำไมใช้ Color Histogram?**
- เร็วกว่า Deep Learning 50-100 เท่า (~1ms vs ~100ms ต่อภาพ)
- ไม่ต้องใช้ GPU สำหรับ Re-ID
- แม่นยำเพียงพอสำหรับจำแนกคนจากสีเสื้อผ้าในระยะสั้น
- สี HSV ทนต่อแสงเงาดีกว่า RGB

In [1]:
import cv2
import numpy as np

# ============================================================
# Feature Extractor แบบเบา (Color Histogram)
# ใช้แทน ResNet50 → เร็วกว่า 50-100 เท่า ไม่ต้องใช้ GPU
# ============================================================

def extract_features(crop_img):
    """ดึง Feature จากภาพคนด้วย Color Histogram (HSV)"""
    if crop_img.size == 0:
        return np.zeros(3000)
    hsv = cv2.cvtColor(crop_img, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [50, 60], [0, 180, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

print("Feature Extractor (Color Histogram) ready!")

Feature Extractor (Color Histogram) ready!


## 3. Pipeline หลัก — ตรวจจับ + นับคนเข้า-ออกประตู

### วิธีทำงาน
1. **YOLOv5n Track** ตรวจจับและติดตามคนในแต่ละเฟรม (ประมวลผลทุก 3 เฟรม เพื่อความเร็ว)
2. **Door Zone (สี่เหลี่ยมกรอบประตู)** กำหนดพื้นที่ประตูเป็นสี่เหลี่ยมผืนผ้า
3. เมื่อจุดศูนย์กลางคนเคลื่อน **เข้า/ออก** พื้นที่สี่เหลี่ยม → ถือว่า crossing
4. **Re-ID** ดึง Color Histogram ไปเทียบกับฐานข้อมูล → ระบุว่าเป็นคนเดิมหรือคนใหม่
5. นับ **Enter / Exit / Return** แยกกัน

### สีกรอบบนวิดีโอ
| สี | ความหมาย |
|---|---|
| เขียว | ตรวจจับได้ แต่ยังไม่เข้า Door Zone |
| น้ำเงิน | `IN P#` — อยู่ใน Door Zone |
| ส้ม | `OUT P#` — ออกจาก Door Zone |
| ม่วง | `RETURN P#` — คนเดิมกลับมาซ้ำ |

### ตั้งค่าที่ปรับได้
- `DOOR_X1, DOOR_Y1, DOOR_X2, DOOR_Y2` — ตำแหน่งกรอบประตู
- `YOLO_IMGSZ` — ขนาดภาพ YOLO (น้อยกว่า = เร็วกว่า)
- `PROCESS_EVERY_N` — ประมวลผลทุก N เฟรม (มากกว่า = เร็วกว่า)
- `SIMILARITY_THRESHOLD` — ค่าความคล้ายขั้นต่ำสำหรับ Re-ID

In [ ]:
import cv2
from collections import deque
from tqdm.notebook import tqdm
from ultralytics import YOLO
import numpy as np

# ============================================================
# ตั้งค่า Door Zone (สี่เหลี่ยมกรอบประตู)
# ============================================================
DOOR_X1, DOOR_Y1 = 100, 50     # มุมซ้ายบน ของกรอบประตู
DOOR_X2, DOOR_Y2 = 550, 950    # มุมขวาล่าง ของกรอบประตู
HISTORY_LENGTH = 10
COOLDOWN_FRAMES = 15           # รอ N เฟรมหลัง crossing ก่อนนับใหม่ (ป้องกันสั่น)

# ============================================================
# ตั้งค่า Performance
# ============================================================
YOLO_IMGSZ = 480
PROCESS_EVERY_N = 3

# ============================================================
# โหลดโมเดล YOLO
# ============================================================
yolo_model = YOLO("yolov5n.pt")

# ============================================================
# ฐานข้อมูล Re-ID
# ============================================================
database = {}
next_global_id = 1
SIMILARITY_THRESHOLD = 0.6

def cosine_sim(a, b):
    dot = np.dot(a, b)
    norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)
    return dot / (norm_a * norm_b) if norm_a > 0 and norm_b > 0 else 0

def match_person(current_embedding):
    global next_global_id
    best_match_id = None
    max_sim = -1
    for gid, saved_embeddings in database.items():
        sims = [cosine_sim(current_embedding, emb) for emb in saved_embeddings]
        avg_sim = np.mean(sims)
        if avg_sim > max_sim:
            max_sim = avg_sim
            best_match_id = gid

    if max_sim >= SIMILARITY_THRESHOLD:
        database[best_match_id].append(current_embedding)
        if len(database[best_match_id]) > 5:
            database[best_match_id] = database[best_match_id][-5:]
        return best_match_id, max_sim
    else:
        new_id = next_global_id
        database[new_id] = [current_embedding]
        next_global_id += 1
        return new_id, 1.0

# ============================================================
# ฟังก์ชันตรวจจับเข้า-ออก Door Zone (สี่เหลี่ยม)
# ============================================================
def is_in_door_zone(cx, cy):
    return DOOR_X1 <= cx <= DOOR_X2 and DOOR_Y1 <= cy <= DOOR_Y2

# ============================================================
# ตัวแปรสถานะ
# ============================================================
track_zone_status = {}    # { track_id: True/False } อยู่ใน/นอก zone
track_global_id = {}      # { track_id: global_id }
crossing_cooldown = {}    # { track_id: frame_idx ที่ crossing ล่าสุด }

enter_count = 0           # จำนวนครั้งเข้า (รวมกลับมาซ้ำ)
exit_count = 0            # จำนวนครั้งออก
return_count = 0          # จำนวนครั้งที่คนเคยเข้าแล้วกลับมาอีก
person_has_entered = set() # global_id ที่เคยเข้าไปแล้ว
last_annotations = []

# ============================================================
# Helper functions สำหรับวาดภาพ
# ============================================================
def draw_door_zone(frame):
    """วาดสี่เหลี่ยมกรอบประตู (semi-transparent + เส้นขอบ)"""
    overlay = frame.copy()
    cv2.rectangle(overlay, (DOOR_X1, DOOR_Y1), (DOOR_X2, DOOR_Y2), (0, 0, 255), -1)
    cv2.addWeighted(overlay, 0.12, frame, 0.88, 0, frame)
    cv2.rectangle(frame, (DOOR_X1, DOOR_Y1), (DOOR_X2, DOOR_Y2), (0, 0, 255), 2)
    cv2.putText(frame, "Door Zone", (DOOR_X1 + 10, DOOR_Y1 + 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

def draw_hud(frame):
    draw_door_zone(frame)
    cv2.putText(frame, f"ENTER: {enter_count}  (Return: {return_count})", (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 0, 0), 3)
    cv2.putText(frame, f"EXIT: {exit_count}", (20, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 165, 255), 3)
    cv2.putText(frame, f"UNIQUE PEOPLE: {len(database)}", (20, 130),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)

def draw_boxes(frame, annotations):
    for x1, y1, x2, y2, label, color, cx, cy in annotations:
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        cv2.circle(frame, (cx, cy), 4, color, -1)

# ============================================================
# เริ่มประมวลผลวิดีโอ
# ============================================================
video_path = r"entrance.mov"
cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
pbar = tqdm(total=total_frames, desc="Processing Video", unit="frame")
out = cv2.VideoWriter('output.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

frame_idx = 0
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    frame_idx += 1

    # === Skipped frame ===
    if PROCESS_EVERY_N > 1 and frame_idx % PROCESS_EVERY_N != 0:
        draw_hud(frame)
        draw_boxes(frame, last_annotations)
        out.write(frame)
        pbar.update()
        continue

    # === ประมวลผลเต็มรูปแบบ ===
    results = yolo_model.track(frame, persist=True, classes=[0], verbose=False, imgsz=YOLO_IMGSZ)
    current_annotations = []

    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.int().cpu().tolist()
        track_ids = results[0].boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = box
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame_width, x2), min(frame_height, y2)

            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            curr_inside = is_in_door_zone(cx, cy)

            # ตรวจจับการเข้า-ออก Door Zone
            crossed = None
            if track_id in track_zone_status:
                prev_inside = track_zone_status[track_id]
                if not prev_inside and curr_inside:
                    crossed = "enter"   # นอก → ใน = เข้า
                elif prev_inside and not curr_inside:
                    crossed = "exit"    # ใน → นอก = ออก

            track_zone_status[track_id] = curr_inside

            # ทำ Re-ID เมื่อ crossing เกิดขึ้น (ผ่าน cooldown)
            if crossed:
                can_cross = (track_id not in crossing_cooldown or
                             (frame_idx - crossing_cooldown[track_id]) >= COOLDOWN_FRAMES)
                if can_cross:
                    crossing_cooldown[track_id] = frame_idx
                    crop_img = frame[y1:y2, x1:x2]
                    if crop_img.size > 0:
                        current_embedding = extract_features(crop_img)
                        global_id, confidence = match_person(current_embedding)
                        track_global_id[track_id] = global_id

                        if crossed == "enter":
                            enter_count += 1
                            if global_id in person_has_entered:
                                return_count += 1  # คนเดิมกลับมาซ้ำ
                            person_has_entered.add(global_id)
                        elif crossed == "exit":
                            exit_count += 1

            # กำหนดสีและ label
            color = (0, 255, 0)      # เขียว = ยังไม่เคย crossing
            label = f"T{track_id}"

            if track_id in track_global_id:
                gid = track_global_id[track_id]
                is_returning = gid in person_has_entered
                if track_id in track_zone_status and track_zone_status[track_id]:
                    if is_returning:
                        color = (255, 0, 255)   # ม่วง = คนเดิมกลับมา
                        label = f"RETURN P{gid}"
                    else:
                        color = (255, 0, 0)     # น้ำเงิน = อยู่ใน zone
                        label = f"IN P{gid}"
                else:
                    color = (0, 165, 255)       # ส้ม = ออกจาก zone
                    label = f"OUT P{gid}"

            current_annotations.append((x1, y1, x2, y2, label, color, cx, cy))

    last_annotations = current_annotations
    draw_hud(frame)
    draw_boxes(frame, current_annotations)
    out.write(frame)
    pbar.update()

cap.release()
pbar.close()
out.release()
print(f"Done! Enter: {enter_count} (Return: {return_count}), Exit: {exit_count}, Unique: {len(database)}")